# Simple Deep NN model for MNIST

In [7]:
# !pip install torch torchvision tqdm

In [1]:
import torch
import torch.nn.functional as F
from torch import nn

In [2]:
# Linear -> ReLu ... (x 4 hidden layers) -> Softmax (Output layer) (Multiclass classification)
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32) # Hidden layer 1, input_dim = flattened pixels of image inputs
        self.fc2 = nn.Linear(32, 64) # Hidden layer 2
        self.fc3 = nn.Linear(64, 128) # Hidden layer 3
        self.fc4 = nn.Linear(128, 64) # Hidden layer 4
        self.fc5 = nn.Linear(64, output_dim) # Output layer, output_dim = number of classes output

    def forward(self, x):
        x = F.relu(self.fc1(x)) # Activation after hidden layer 1
        x = F.relu(self.fc2(x)) # Activation after hidden layer 2
        x = F.relu(self.fc3(x)) # Activation after hidden layer 3
        x = F.relu(self.fc4(x)) # Activation after hidden layer 4
        x = self.fc5(x) # Output layer
        return x


# Data Loading

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [4]:
BATCH_SIZE = 64
train_dataset = datasets.MNIST(root='dataset/', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='dataset/', train=False, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
input_dim = train_dataset[0][0].numel()
no_classes = 10

# Train

In [6]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/nn_experiment")

2026-06-29 14:43:45.260369: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 14:43:45.326716: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-29 14:43:46.709118: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 14:43:46.709535: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31]

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralNetwork(input_dim=input_dim, output_dim=no_classes).to(device)

In [8]:
criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device).reshape(x.shape[0], -1)
            y = y.to(device)

            scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    model.train()

    return num_correct / num_samples

In [11]:
epochs = 10
step = 0

for epoch in range(epochs):
    running_loss = 0.0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device).reshape(data.shape[0], -1)
        targets = targets.to(device)

        # Forward propagation
        scores = model(data)
        loss = criterion(scores, targets)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Log batch loss
        writer.add_scalar("Training Loss / Batch", loss.item(), step)

        step += 1

    # Average loss for this epoch
    avg_epoch_loss = running_loss / len(train_loader)

    # Compute accuracy after each epoch
    train_acc = check_accuracy(train_loader, model)
    test_acc = check_accuracy(test_loader, model)

    # Log epoch metrics
    writer.add_scalar("Training Loss / Epoch", avg_epoch_loss, epoch)
    writer.add_scalar("Training Accuracy", train_acc, epoch)
    writer.add_scalar("Test Accuracy", test_acc, epoch)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_epoch_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )

writer.close()

Epoch [1/10], Loss: 0.0890, Train Acc: 0.9750, Test Acc: 0.9655
Epoch [2/10], Loss: 0.0785, Train Acc: 0.9797, Test Acc: 0.9684
Epoch [3/10], Loss: 0.0716, Train Acc: 0.9830, Test Acc: 0.9682
Epoch [4/10], Loss: 0.0667, Train Acc: 0.9835, Test Acc: 0.9699
Epoch [5/10], Loss: 0.0601, Train Acc: 0.9842, Test Acc: 0.9685
Epoch [6/10], Loss: 0.0544, Train Acc: 0.9857, Test Acc: 0.9700
Epoch [7/10], Loss: 0.0514, Train Acc: 0.9810, Test Acc: 0.9669
Epoch [8/10], Loss: 0.0465, Train Acc: 0.9851, Test Acc: 0.9674
Epoch [9/10], Loss: 0.0433, Train Acc: 0.9875, Test Acc: 0.9706
Epoch [10/10], Loss: 0.0420, Train Acc: 0.9913, Test Acc: 0.9705
